# Workit RAG 평가(eval) on Colab (GPU) — 단계적(Staged) Sweep, reranker 단일화 버전

이 노트북은 로컬에서 만든 RAG 파이프라인(JoRAG/HoRAG)을 Colab GPU에서
돌리기 위한 것입니다. 로컬 Qdrant 서버에 접속하는 대신, 이미 임베딩이 끝난
파일들을 Colab에 직접 업로드해서 **Colab 세션 안에서 로컬(파일 기반) Qdrant를
새로 만들고** 그 위에서 단계적 sweep을 실행합니다.

## 이전 버전과 달라진 점
- **reranker를 `bge-reranker-v2-m3` 하나로 통합** (ko-reranker/reranker1 계열 제거)
- `alpha`, `rrf_k`, `fetch_k`, `rerank_k`, `use_cross_refs`, `max_cross_ref_tokens`를
  **완전 교차곱이 아니라 단계적(staged)으로** sweep — 조합 폭발(2280개까지 감)을 피하기 위함
  - 0단계: reranker OFF baseline
  - 1단계: alpha × rrf_k (거의 공짜인 축 먼저 넓게)
  - 2단계: fetch_k × rerank_k (1단계 최적값 고정, 비용 드는 축만 따로)
  - 3단계 (HoRAG만): use_cross_refs on/off + max_cross_ref_tokens (2단계 최적값 고정)
- `top_k`는 sweep 축이 아니라, 한 번 검색한 결과에서 Recall@1/3/5/10을 전부 계산

## 실행 전 체크리스트
1. **런타임 → 런타임 유형 변경 → GPU(T4)** 로 바꾸세요.
2. 왼쪽 파일 탐색기(📁)에서 `/content/` 에 아래 파일들을 업로드하세요.
   - `vectors_jo_fixedid.npz`
   - `vectors_ho_fixedid.npz`
   - `sparse_weights_jo_fixedid.json`
   - `sparse_weights_ho_fixedid.json`
   - `chunks_jo_fixedid.json`
   - `chunks_ho_fixedid.json`
   - `eval_set.json` (평가셋 — 50개든 100개든 파일명이 다르면 아래 셀에서 경로만 바꿔주세요)
3. 위 파일들이 다 올라간 뒤 셀을 위에서부터 순서대로 실행하세요.

## 노트북 구성
1. 패키지 설치 / GPU 확인
2. 파이프라인 코드 작성 (`yoonha_law_rag.py`, `yoonha_colab_upsert.py`)
3. 업로드 파일 존재 확인
4. 로컬 Qdrant 생성 + upsert
5. 단계적(Staged) sweep 실행 (`yoonha_rag_eval_staged.py`)
6. 결과 확인 + 다운로드


## 1. 패키지 설치 / GPU 확인

In [1]:
!pip install qdrant-client FlagEmbedding transformers torch pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.0 MB/s eta 0:00:00


In [2]:
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU가 안 잡혔습니다. 런타임 → 런타임 유형 변경 → GPU(T4)로 바꾼 뒤 다시 실행하세요.")

CUDA 사용 가능: True
GPU: Tesla T4


In [3]:
!nvidia-smi

Sun Jul  5 13:24:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. 파이프라인 코드 작성

아래 2개 셀은 `%%writefile`로 `/content/`에 실제 `.py` 파일을 만듭니다.
`yoonha_law_rag.py`는 reranker 단일화 + `rrf_k`/`max_cross_ref_tokens` 축이
추가된 개정판입니다.

In [4]:
%%writefile /content/yoonha_law_rag.py
"""
Workit - 계약서 검토 RAG 파이프라인 (2-flow, sweep 가능한 버전 + 캐싱)
파일명: rag/yoonha_law_rag.py

전체 흐름 (다이어그램 기준):
  1) JoRAG   : law_kb_jo_fixedid 에서 "조" 단위로 직접 검색 (넓은 단위, 청킹 크기 큼)
  2) HoRAG   : law_kb_ho_fixedid 에서 "호/목/세목"까지 쪼갠 세부 단위로 검색
               → 후보들의 cross_refs로 관련 조항 추가 확장 (별도 xref 컬렉션 없음,
                 cross_refs가 이미 ho payload 안에 들어있음)
               → 항상 parent_chunk_id로 "조" 텍스트를 fetch해서 최종 출력 단위를
                 조로 통일 (하위 단위는 검색에만 쓰고 최종 결과로는 노출 안 함)
  두 flow 모두 계약서 조항 1개당 결과를 병합해서 list[ClauseResult]로 반환.

이번 개정 (2026-07-04, reranker 단일화 + sweep 축 확장):
  - reranker를 bge-reranker-v2-m3 하나로 고정. 기존 reranker1(ko-reranker)/
    reranker2 이원 구조를 전부 걷어내고 reranker 파라미터 하나로 통합했다.
    JoRAG/HoRAG 둘 다 이 reranker 하나만 쓴다.
  - RRF_K가 _hybrid_search 안에 하드코딩(60)돼 있던 걸 함수 인자로 뺐다.
    alpha와 마찬가지로 RRF 합산 단계에서만 쓰이므로, raw_search 캐시를
    전혀 건드리지 않고 sweep 가능하다 (사실상 공짜 축).
  - HoRAG의 cross-ref 확장에 토큰 예산(max_cross_ref_tokens) 개념을 추가했다.
    "몇 개까지 추가할지"가 아니라 "reranker가 실제로 받는 총 토큰이 예산을
    넘지 않는 선까지 추가"하는 방식이다. 원본 후보(hybrid search로 직접 찾은
    것)는 무조건 유지하고, cross-ref로 딸려온 추가 후보만 트리밍 대상이다.
    우선순위는 "어떤 원본 후보가 인용한 것인지"의 rrf_score(source_score)가
    높은 순.
  - 이 트리밍은 parent fetch *이후*에 수행한다. cross-ref 확장 시점(parent
    fetch 전)에는 아직 호/목 단위의 짧은 텍스트라서 토큰 수를 재봤자 의미가
    없다 — parent fetch로 조 전체 텍스트로 바뀐 뒤에야 실제로 reranker가
    보게 될 길이를 알 수 있다.
  - top_k는 기존과 동일하게 "이미 순위 매겨진 후보 리스트를 자르기만" 하는
    구조라 별도 캐싱/재검색이 필요 없다. sweep 스크립트에서는 top_k=1/3/5/10을
    동일한 한 번의 검색 결과에서 전부 계산하면 된다 (재검색 불필요).

sweep 가능하게 바뀐 점 (이전 버전과 공통):
  - alpha, use_reranker, rerank_k, fetch_k, top_k, rrf_k, max_cross_ref_tokens을
    전부 모듈 상수가 아니라 함수 인자로 뺐다.
  - 모듈 상수는 "sweep 안 할 때 쓰는 기본값" 역할만 한다 (DEFAULT_* 접두사).

캐싱 구조:
  - alpha, rrf_k는 RRF 가중치 합산에만 쓰이고, 임베딩/Qdrant raw 검색 결과와는
    무관하다. 그래서 이 두 값을 sweep할 때 (collection, query_text, fetch_k)별
    raw 검색 결과를 한 번만 구하고, 조합마다는 캐시된 raw 결과로 RRF 점수만
    다시 계산한다.
  - reranker 점수는 (reranker_name, query_text, chunk_id) 쌍에만 의존한다.
    alpha/rrf_k/max_cross_ref_tokens이 바뀌면 어떤 chunk_id가 rerank_k 안에
    들어오는지는 달라질 수 있지만, 같은 chunk_id에 대한 점수 자체는 항상 같다.
    그래서 미스(cache miss)만 골라서 계산하고 나머지는 캐시에서 꺼내 쓴다.
  - parent fetch(조 텍스트)와 cross-ref 확장(scroll 조회)도 chunk_id -> payload
    조회라 alpha/rrf_k/쿼리와 무관하게 전역 캐시가 가능하다. 여러 조항이 같은
    법 조항을 참조하는 경우가 많아서 쿼리 간에도 재사용된다.
  - fetch_k는 예외다 — raw_search 캐시 키에 fetch_k가 포함돼 있어서, fetch_k
    값이 바뀌면 Qdrant 검색 자체가 다시 돈다 (alpha/rrf_k처럼 공짜가 아님).
  - cache=None으로 호출하면 캐싱 없이 기존과 완전히 동일하게 동작한다.

컬렉션:
  - JoRAG : law_kb_jo_fixedid (조 단위, parent 없음)
  - HoRAG : law_kb_ho_fixedid (호/목/세목 단위 + cross_refs, parent fetch → 조 텍스트)

공통 출력: list[ClauseResult]  ← 항상 조 단위로 반환
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass, field, asdict
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient
from qdrant_client.models import SparseVector, Filter, FieldCondition, MatchValue

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 경로 / 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
_THIS_DIR     = Path(__file__).resolve().parent
_DATA_DIR     = _THIS_DIR.parent / "data"
LAWS_REF_PATH = _DATA_DIR / "hn_seed" / "law_refs.json"

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

COLLECTION_JO = "law_kb_jo_fixedid"
COLLECTION_HO = "law_kb_ho_fixedid"
# HoXrefRAG는 별도 컬렉션 없이 COLLECTION_HO를 그대로 씀 (cross_refs가 payload에 이미 있음)

EMBED_MODEL = "BAAI/bge-m3"

# sweep 안 할 때 쓰는 기본값. 실제 최적값은 yoonha_rag_eval.py로 찾는다.
DEFAULT_FETCH_K              = 50
DEFAULT_RERANK_K             = 10
DEFAULT_TOP_K                = 10
DEFAULT_ALPHA                = 0.5    # 1.0 = dense only, 0.0 = sparse only
DEFAULT_RRF_K                = 60     # RRF 공식의 스무딩 상수 (기존 하드코딩값)
DEFAULT_MAX_CROSS_REF_TOKENS = None   # None = 트리밍 없음 (기존과 동일 동작)

# reranker는 이제 이거 하나만 쓴다 (ko-reranker/reranker1 계열 제거).
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 확정된 최적 조합
# reranker가 하나로 통합됐으므로, use_reranker on/off만 남는다.
# fetch_k / rerank_k / rrf_k / max_cross_ref_tokens의 최종값은
# yoonha_rag_eval.py 재sweep 결과로 다시 채워 넣을 것.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BEST_JO_CONFIG = dict(
    use_reranker=True,
    alpha=DEFAULT_ALPHA,
)

BEST_HO_CONFIG = dict(
    use_reranker=True,
    alpha=DEFAULT_ALPHA,
    use_cross_refs=True,
    max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Sweep 캐시
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class SweepCache:
    """
    alpha × rrf_k × reranker on/off × fetch_k × ... sweep에서 재사용 가능한
    중간 계산 결과를 담는 캐시.

    - embed        : query_text -> (dense_vec, sparse_vec)
                      (임베딩은 collection/alpha/rrf_k와 무관 — jo/ho variant 간에도 공유됨)
    - raw_search   : (collection, query_text, fetch_k) -> (dense_points, sparse_points)
                      (alpha/rrf_k와 무관한 Qdrant 원본 검색 결과. RRF 합산만 alpha/rrf_k
                       조합마다 다시 함. fetch_k가 바뀌면 캐시 키 자체가 달라짐 — 재검색 필요.)
    - scroll       : (collection, chunk_id) -> payload
                      (parent fetch / cross-ref 확장에서 쓰는 단건 조회. 여러 쿼리에서
                       같은 법 조항을 참조하면 자동으로 재사용됨)
    - rerank_score : (reranker_name, query_text, chunk_id) -> score
                      (reranker cross-encoder 점수. alpha/rrf_k/max_cross_ref_tokens이
                       달라져도 같은 chunk_id에 대한 점수는 항상 같으므로 미스만 계산)

    cache=None으로 함수를 호출하면 캐싱 없이 기존과 동일하게 동작한다.
    """
    embed        : dict = field(default_factory=dict)
    raw_search   : dict = field(default_factory=dict)
    scroll       : dict = field(default_factory=dict)
    rerank_score : dict = field(default_factory=dict)

    def stats(self) -> dict:
        return {
            "embed_cached":      len(self.embed),
            "raw_search_cached": len(self.raw_search),
            "scroll_cached":     len(self.scroll),
            "rerank_cached":     len(self.rerank_score),
        }


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Cross-encoder Reranker
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class CrossEncoderReranker:
    """
    transformers AutoModel 기반 Cross-encoder reranker.
    FlagReranker 대체용 — 최신 transformers 호환.
    """

    def __init__(self, model_name: str, device: str = "cpu"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        self.device = device

    def compute_score(
        self,
        pairs     : list[list[str]],
        batch_size: int  = 32,
        normalize : bool = True,
    ) -> list[float]:
        all_scores: list[float] = []

        for i in range(0, len(pairs), batch_size):
            batch   = pairs[i : i + batch_size]
            encoded = self.tokenizer(
                [p[0] for p in batch],
                [p[1] for p in batch],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            encoded = {k: v.to(self.device) for k, v in encoded.items()}

            with torch.no_grad():
                logits = self.model(**encoded).logits

            scores = logits.squeeze(-1) if logits.shape[-1] == 1 else logits[:, 1]
            if normalize:
                scores = torch.sigmoid(scores)

            all_scores.extend(scores.cpu().tolist())

        return all_scores

    def count_tokens(self, text: str) -> int:
        """토큰 예산 계산용. truncation 없이 실제 토큰 수를 그대로 센다."""
        return len(self.tokenizer.encode(text, add_special_tokens=False))


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 클래스
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class LawRef:
    """검색된 법령 조문 1건."""
    chunk_id   : str
    article    : str
    category   : str
    law_name   : str
    chunk_text : str
    score      : float
    is_risk_ref: bool
    parent_id  : str = ""
    cross_refs : list[str] = field(default_factory=list)  # HoRAG(xref 확장) 전용


@dataclass
class ClauseResult:
    """계약서 조항 1건의 검색 결과 — 항상 조 단위."""
    clause_number: str
    clause_text  : str
    page         : int            = 0
    bbox         : dict | None    = None
    law_refs     : list[LawRef]   = field(default_factory=list)
    categories   : list[str]      = field(default_factory=list)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 공통 유틸
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_laws_ref(path: Path = LAWS_REF_PATH) -> dict[str, dict]:
    if not path.exists():
        print(f"  ⚠️  laws_ref.json 없음: {path}")
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_embed_model(model_name: str = EMBED_MODEL, use_fp16: bool = True) -> BGEM3FlagModel:
    print(f"📦 임베딩 모델 로드: {model_name}")
    return BGEM3FlagModel(model_name, use_fp16=use_fp16)


def load_reranker(device: str = "cpu") -> CrossEncoderReranker:
    """
    reranker는 로드 비용이 커서 sweep 중에는 한 번만 로드해두고,
    실제로 쓸지 말지는 각 검색 함수의 use_reranker 플래그로 토글한다.
    """
    print(f"📦 Re-ranker 로드: {RERANKER_MODEL}")
    return CrossEncoderReranker(RERANKER_MODEL, device=device)


def derive_jo_id(chunk_id: str) -> str:
    """
    ho-level chunk_id 문자열에서 그 조에 해당하는 jo-level chunk_id를 직접 역산한다.

    ho id 형식: {prefix}_{장}_{절}_{조}_{항}_{호}_{목}_{세목} (8토큰 고정)
    jo id 형식: 일반 법령은 {prefix}_{장}_{절}_{조} (앞 4토큰),
                PYG(예규)는 조가 없어 항이 anchor이므로 {prefix}_{장}_{절}_{조=0}_{항} (앞 5토큰).

    주의: payload의 parent_chunk_id 필드는 이 용도로 쓰면 안 된다. 실제 데이터를
    검증해보면 parent_chunk_id는 JO 컬렉션이 아니라 HO 컬렉션 자기 자신 안의 다른
    chunk(조 단위로 롤업된 chunk, 혹은 중간 단계인 호/목)를 가리키고 있어서 JO
    컬렉션 chunk_id와 절대 일치하지 않는다 (검증: ho 7640개 중 parent_chunk_id가
    JO chunk_id와 일치한 건 0개). 반면 이 함수처럼 chunk_id 자신의 앞쪽 토큰만
    잘라내는 방식은 ho 7640개 전부 100% 올바른 jo_id로 매핑된다 (leaf 청크든
    조 단위 롤업 청크든 동일하게 성립).
    """
    tokens = chunk_id.split("_")
    if tokens[0] == "PYG":
        return "_".join(tokens[:5])
    return "_".join(tokens[:4])


def get_vectors(
    text : str,
    model: BGEM3FlagModel,
    cache: SweepCache | None = None,
) -> tuple[list[float], dict[int, float]]:
    """
    BGE-M3 dense/sparse 임베딩. collection이나 alpha/rrf_k와 무관하므로
    query_text만으로 캐시 가능 (jo/ho variant 간에도 재사용됨).
    """
    if cache is not None and text in cache.embed:
        return cache.embed[text]

    output = model.encode(
        [text],
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    dense_vec       = output["dense_vecs"][0].tolist()
    lexical_weights = output["lexical_weights"][0]

    sparse_vec: dict[int, float] = {}
    for token_str, weight in lexical_weights.items():
        token_id = model.tokenizer.convert_tokens_to_ids(token_str)
        if isinstance(token_id, int):
            sparse_vec[token_id] = sparse_vec.get(token_id, 0.0) + float(weight)

    if cache is not None:
        cache.embed[text] = (dense_vec, sparse_vec)

    return dense_vec, sparse_vec


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 계약서 청킹 (조 단위 출력)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def chunk_contract(text: str) -> list[dict]:
    """계약서를 조 단위로 청킹."""
    HANG_MAP = {c: i + 1 for i, c in enumerate("①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮")}
    HO_SPLIT_PATTERN = r"(?:^|\s)(\d{1,2}\.\s)"

    text = text.strip()
    header_pattern = re.compile(r"제(\d+)조(?:의(\d+))?\s*\(([^)]*)\)")
    raw_matches    = list(header_pattern.finditer(text))

    candidates = []
    for m in raw_matches:
        prefix = text[max(0, m.start() - 5):m.start()]
        if re.search(r"법\s*$", prefix):
            continue
        num           = int(m.group(1))
        sub           = m.group(2)
        clause_number = f"제{m.group(1)}조" + (f"의{sub}" if sub else "")
        candidates.append((num, clause_number, m.start()))

    header_spans = []
    last_num = 0
    for num, clause_number, start in candidates:
        if num >= last_num and num <= last_num + 5:
            header_spans.append((clause_number, start))
            last_num = num

    def split_into_ho(parent_number: str, unit_text: str) -> list[dict]:
        ho_splits = re.split(HO_SPLIT_PATTERN, unit_text)
        if len(ho_splits) <= 1:
            return [{"clause_number": parent_number, "clause_text": unit_text}]

        head   = ho_splits[0].strip()
        chunks = []
        if head:
            chunks.append({"clause_number": parent_number, "clause_text": head})

        k, last_ho_num = 1, 0
        while k < len(ho_splits) - 1:
            marker       = ho_splits[k].strip()
            ho_num_match = re.match(r"(\d{1,2})\.", marker)
            ho_num       = int(ho_num_match.group(1)) if ho_num_match else (k // 2 + 1)
            ho_body      = ho_splits[k + 1].strip() if k + 1 < len(ho_splits) else ""

            if ho_num == last_ho_num + 1 and ho_body:
                chunks.append({
                    "clause_number": f"{parent_number}제{ho_num}호",
                    "clause_text":   re.sub(r"\s+", " ", f"{marker} {ho_body}").strip(),
                })
                last_ho_num = ho_num
            elif ho_body:
                if chunks:
                    chunks[-1]["clause_text"] += f" {marker} {ho_body}"
                else:
                    chunks.append({"clause_number": parent_number, "clause_text": f"{marker} {ho_body}"})
            k += 2

        return chunks if chunks else [{"clause_number": parent_number, "clause_text": unit_text}]

    clauses = []
    for idx, (clause_number, start) in enumerate(header_spans):
        end       = header_spans[idx + 1][1] if idx + 1 < len(header_spans) else len(text)
        raw_block = text[start:end].strip()

        m          = header_pattern.match(raw_block)
        raw_header = m.group(0) if m else clause_number
        body       = raw_block[m.end():].strip() if m else raw_block

        if not body:
            continue

        hang_splits = re.split(r"([①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮])", body)

        if len(hang_splits) <= 1:
            clause_text = re.sub(r"\s+", " ", f"{raw_header} {body}").strip()
            clauses.extend(split_into_ho(clause_number, clause_text))
        else:
            j = 1
            while j < len(hang_splits) - 1:
                hang_char   = hang_splits[j]
                hang_body   = hang_splits[j + 1].strip() if j + 1 < len(hang_splits) else ""
                hang_num    = HANG_MAP.get(hang_char, j)
                if hang_body:
                    hang_text = re.sub(r"\s+", " ", f"{raw_header} {hang_char}{hang_body}").strip()
                    clauses.extend(split_into_ho(f"{clause_number}제{hang_num}항", hang_text))
                j += 2

    if not clauses:
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        clauses = [
            {"clause_number": f"단락{i + 1}", "clause_text": para}
            for i, para in enumerate(paragraphs)
        ]

    return clauses


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 공통 검색 / 리랭크 / parent fetch
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _hybrid_search(
    clause_text: str,
    client     : QdrantClient,
    model      : BGEM3FlagModel,
    collection : str,
    fetch_k    : int,
    alpha      : float,
    rrf_k      : int = DEFAULT_RRF_K,
    cache      : SweepCache | None = None,
) -> list[dict]:
    """
    Dense + Sparse 하이브리드 검색 (수동 RRF). alpha=1.0이면 dense만, 0.0이면 sparse만.

    alpha와 rrf_k는 아래 RRF 합산 단계에서만 쓰이므로, Qdrant raw 검색 결과
    (dense_results, sparse_results)는 (collection, clause_text, fetch_k)가
    같으면 alpha/rrf_k와 무관하게 재사용 가능 — cache가 있으면 캐시에서 꺼내온다.
    (fetch_k가 바뀌면 캐시 키 자체가 달라지므로 이 값은 sweep 시 재검색이 발생한다.)
    """
    cache_key = (collection, clause_text, fetch_k)

    if cache is not None and cache_key in cache.raw_search:
        dense_results, sparse_results = cache.raw_search[cache_key]
    else:
        dense_vec, sparse_vec = get_vectors(clause_text, model, cache)
        indices = list(sparse_vec.keys())
        values  = list(sparse_vec.values())

        try:
            dense_results = client.query_points(
                collection_name=collection,
                query=dense_vec,
                using="dense",
                limit=fetch_k,
                with_payload=True,
            ).points

            sparse_results = client.query_points(
                collection_name=collection,
                query=SparseVector(indices=indices, values=values),
                using="sparse",
                limit=fetch_k,
                with_payload=True,
            ).points

        except Exception as e:
            print(f"  ⚠️  sparse 검색 실패, dense만 사용: {e}")
            dense_results = client.query_points(
                collection_name=collection,
                query=dense_vec,
                using="dense",
                limit=fetch_k,
                with_payload=True,
            ).points
            sparse_results = []

        if cache is not None:
            cache.raw_search[cache_key] = (dense_results, sparse_results)

    scores: dict[str, dict] = {}

    for rank, point in enumerate(dense_results, 1):
        cid = point.payload.get("chunk_id", str(point.id))
        scores[cid] = {
            "payload":     point.payload,
            "dense_rank":  rank,
            "sparse_rank": len(dense_results) + 1,
        }

    for rank, point in enumerate(sparse_results, 1):
        cid = point.payload.get("chunk_id", str(point.id))
        if cid in scores:
            scores[cid]["sparse_rank"] = rank
        else:
            scores[cid] = {
                "payload":     point.payload,
                "dense_rank":  len(sparse_results) + 1,
                "sparse_rank": rank,
            }

    results = []
    for cid, info in scores.items():
        rrf_score = (
            alpha         * (1 / (rrf_k + info["dense_rank"]))
            + (1 - alpha) * (1 / (rrf_k + info["sparse_rank"]))
        )
        results.append({
            "chunk_id"    : cid,
            "payload"     : info["payload"],
            "rrf_score"   : rrf_score,
            "is_cross_ref": False,   # 원본 후보 표시 (cross-ref 트리밍에서 구분용)
        })

    results.sort(key=lambda x: x["rrf_score"], reverse=True)
    return results


def _rerank(
    query        : str,
    candidates   : list[dict],
    reranker     : CrossEncoderReranker,
    top_k        : int,
    reranker_name: str = "reranker",
    cache        : SweepCache | None = None,
) -> list[dict]:
    """
    reranker_name은 캐시 키 네임스페이스 구분용 (예: "jo", "ho").
    같은 (reranker_name, query, chunk_id) 조합은 alpha/rrf_k가 달라져도 점수가
    동일하므로, cache가 있으면 미스(cache miss)만 계산하고 나머지는 재사용한다.
    """
    if not candidates:
        return []

    if cache is None:
        texts  = [c["payload"].get("text", c["payload"].get("chunk_text", "")) for c in candidates]
        pairs  = [[query, t] for t in texts]
        scores = reranker.compute_score(pairs, normalize=True)
    else:
        scores: list[float | None] = [None] * len(candidates)
        miss_idx: list[int] = []

        for i, c in enumerate(candidates):
            key = (reranker_name, query, c["chunk_id"])
            if key in cache.rerank_score:
                scores[i] = cache.rerank_score[key]
            else:
                miss_idx.append(i)

        if miss_idx:
            miss_texts = [
                candidates[i]["payload"].get("text", candidates[i]["payload"].get("chunk_text", ""))
                for i in miss_idx
            ]
            miss_pairs  = [[query, t] for t in miss_texts]
            miss_scores = reranker.compute_score(miss_pairs, normalize=True)

            for i, s in zip(miss_idx, miss_scores):
                scores[i] = s
                cache.rerank_score[(reranker_name, query, candidates[i]["chunk_id"])] = s

    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [item for _, item in ranked[:top_k]]


def _fetch_parent_texts(
    candidates: list[dict],
    client    : QdrantClient,
    parent_collection: str = COLLECTION_JO,
    cache     : SweepCache | None = None,
) -> list[dict]:
    """
    각 후보 ho chunk_id에서 derive_jo_id()로 조 단위 chunk_id를 역산해,
    그 조 텍스트를 law_kb_jo_fixedid에서 조회해 payload["text"]를 교체한다.

    payload의 parent_chunk_id 필드는 쓰지 않는다 (derive_jo_id 함수 docstring 참고
    — 그 필드는 JO 컬렉션이 아니라 HO 컬렉션 자기 자신을 가리키고 있어서 이 용도로
    쓰면 매번 조회가 실패한다). chunk_id 자신의 앞쪽 토큰만으로 역산하는 방식은
    leaf 청크든 조 단위 롤업 청크든 상관없이 항상 올바른 jo_id를 준다.
    """
    jo_ids = list({
        derive_jo_id(c["payload"].get("chunk_id", c["chunk_id"]))
        for c in candidates
    })

    if not jo_ids:
        return candidates

    parent_texts: dict[str, str] = {}
    to_fetch: list[str] = []

    for jid in jo_ids:
        cache_key = (parent_collection, jid)
        if cache is not None and cache_key in cache.scroll:
            payload = cache.scroll[cache_key]
            parent_texts[jid] = payload.get("text", payload.get("chunk_text", ""))
        else:
            to_fetch.append(jid)

    try:
        for jid in to_fetch:
            results = client.scroll(
                collection_name=parent_collection,
                scroll_filter=Filter(
                    must=[FieldCondition(key="chunk_id", match=MatchValue(value=jid))]
                ),
                limit=1,
                with_payload=True,
                with_vectors=False,
            )
            points = results[0]
            if points:
                p = points[0].payload
                parent_texts[jid] = p.get("text", p.get("chunk_text", ""))
                if cache is not None:
                    cache.scroll[(parent_collection, jid)] = p
    except Exception as e:
        print(f"  ⚠️  parent fetch 실패: {e}")
        return candidates

    updated = []
    for c in candidates:
        jid = derive_jo_id(c["payload"].get("chunk_id", c["chunk_id"]))
        if jid in parent_texts:
            updated_payload         = dict(c["payload"])
            updated_payload["text"] = parent_texts[jid]
            updated.append({**c, "payload": updated_payload})
        else:
            updated.append(c)

    return updated


def _build_law_refs(
    candidates : list[dict],
    laws_ref   : dict[str, dict],
    top_k      : int,
    with_xref  : bool = False,
) -> list[LawRef]:
    """
    top_k는 여기서 "이미 순위 매겨진 candidates를 자르기만" 한다 — 재검색이나
    재계산이 전혀 없으므로, 같은 candidates에 대해 top_k=1/3/5/10을 전부
    별도 비용 없이 뽑아낼 수 있다 (sweep 스크립트에서 활용).
    """
    law_refs: list[LawRef] = []
    for c in candidates[:top_k]:
        payload  = c["payload"]
        chunk_id = payload.get("chunk_id", "")
        ref_meta = laws_ref.get(chunk_id, {})

        law_refs.append(LawRef(
            chunk_id    = chunk_id,
            article     = ref_meta.get("article",  payload.get("article_number", "")),
            category    = ref_meta.get("category", payload.get("category", "")),
            law_name    = payload.get("law_name",  ""),
            chunk_text  = payload.get("text", payload.get("chunk_text", "")),
            score       = round(float(c.get("rrf_score", 0.0)), 4),
            is_risk_ref = bool(payload.get("is_risk_ref", False)),
            parent_id   = payload.get("parent_chunk_id", "") or "",
            cross_refs  = payload.get("cross_refs", []) if with_xref else [],
        ))

    return law_refs


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG 1: JoRAG — 조 단위 검색
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _search_jo(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict],
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    JoRAG: law_kb_jo_fixedid에서 조 단위로 직접 검색.
    parent fetch 없음 — 이미 조 단위가 최상위.
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_JO, fetch_k, alpha, rrf_k, cache)

    if use_reranker and reranker and candidates:
        candidates = _rerank(clause_text, candidates, reranker, rerank_k, "jo", cache)

    return _build_law_refs(candidates, laws_ref, top_k, with_xref=False)


def review_contract_jo(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict] | None = None,
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """JoRAG 메인 인터페이스."""
    if laws_ref is None:
        laws_ref = load_laws_ref()

    clauses = chunk_contract(contract_text)
    results : list[ClauseResult] = []
    print(f"[JoRAG] 총 {len(clauses)}개 청크 검색 중...")

    for i, clause in enumerate(clauses, 1):
        print(f"  [{i}/{len(clauses)}] {clause['clause_number']} ...", end="\r")

        law_refs   = _search_jo(
            clause["clause_text"], client, model, laws_ref,
            reranker, use_reranker,
            top_k, alpha, fetch_k, rerank_k, rrf_k, cache,
        )
        categories = list(dict.fromkeys(r.category for r in law_refs if r.category))

        results.append(ClauseResult(
            clause_number=clause["clause_number"],
            clause_text  =clause["clause_text"],
            law_refs     =law_refs,
            categories   =categories,
        ))

    print(f"\n[JoRAG] ✅ 완료")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG 2: HoRAG — 호/목/세목 단위 검색 + cross_refs 확장 + parent fetch
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _expand_with_cross_refs(
    candidates: list[dict],
    client    : QdrantClient,
    cache     : SweepCache | None = None,
) -> list[dict]:
    """
    각 후보의 cross_refs(같은 law_kb_ho_fixedid payload 안의 필드)에 있는
    chunk_id를 추가 조회. 이미 후보에 있는 chunk_id는 중복 추가하지 않음.

    여기서는 개수/토큰 제한을 걸지 않고 전부 모은다 — 토큰 예산 트리밍은
    parent fetch 이후(_trim_cross_refs_by_token_budget)에서 한다. 이 시점의
    텍스트는 아직 호/목 단위라 parent fetch 후 조 전체 텍스트로 바뀌면
    길이가 크게 달라지기 때문에, 여기서 트리밍하면 실제 reranker가 보게 될
    길이 기준과 안 맞다.

    추가된 항목에는 "어떤 원본 후보가 이 조항을 인용했는지"의 rrf_score를
    source_score로 같이 기록해둔다 — 트리밍 시 우선순위(원 후보 점수가 높을수록
    먼저 살림)로 쓰기 위함. 여러 원본 후보가 같은 chunk_id를 인용하면
    source_score가 가장 높은 것을 기록한다.

    chunk_id -> payload 조회라 쿼리/alpha와 무관 — 여러 조항이 같은 참조를
    가지면 캐시에서 재사용된다.
    """
    existing_ids = {c["chunk_id"] for c in candidates}
    best_source_score: dict[str, float] = {}

    for c in candidates:
        cross_refs = c["payload"].get("cross_refs", [])
        for ref_id in cross_refs:
            if ref_id in existing_ids:
                continue
            prev = best_source_score.get(ref_id, float("-inf"))
            if c["rrf_score"] > prev:
                best_source_score[ref_id] = c["rrf_score"]

    ref_ids_total = list(best_source_score.keys())
    if not ref_ids_total:
        return candidates

    payload_by_ref: dict[str, dict] = {}
    to_fetch: list[str] = []

    for ref_id in ref_ids_total:
        cache_key = (COLLECTION_HO, ref_id)
        if cache is not None and cache_key in cache.scroll:
            payload_by_ref[ref_id] = cache.scroll[cache_key]
        else:
            to_fetch.append(ref_id)

    try:
        for ref_id in to_fetch:
            results = client.scroll(
                collection_name=COLLECTION_HO,
                scroll_filter=Filter(
                    must=[FieldCondition(key="chunk_id", match=MatchValue(value=ref_id))]
                ),
                limit=1,
                with_payload=True,
                with_vectors=False,
            )
            points = results[0]
            if points:
                p = points[0].payload
                payload_by_ref[ref_id] = p
                if cache is not None:
                    cache.scroll[(COLLECTION_HO, ref_id)] = p
    except Exception as e:
        print(f"  ⚠️  cross_ref fetch 실패: {e}")

    extra_chunks = []
    for ref_id, p in payload_by_ref.items():
        extra_chunks.append({
            "chunk_id"    : ref_id,
            "payload"     : p,
            "rrf_score"   : 0.0,                       # 리랭크에서 점수 재계산됨
            "is_cross_ref": True,                       # 트리밍 대상 표시
            "source_score": best_source_score[ref_id],   # 우선순위 기준
        })

    return candidates + extra_chunks


def _trim_cross_refs_by_token_budget(
    candidates: list[dict],
    reranker  : CrossEncoderReranker,
    max_tokens: int | None,
) -> list[dict]:
    """
    parent fetch 이후(조 전체 텍스트로 교체된 뒤)에 호출한다.

    원본 후보(is_cross_ref=False, hybrid search로 직접 찾은 것)는 무조건
    유지하고, cross-ref로 추가된 후보만 source_score(이 조항을 인용한 원본
    후보의 rrf_score) 내림차순으로 정렬해서, 누적 토큰 수가 max_tokens를
    넘기 전까지만 살린다.

    max_tokens=None이면 트리밍 없이 그대로 반환 (기존 동작과 동일).
    """
    if max_tokens is None:
        return candidates

    originals = [c for c in candidates if not c.get("is_cross_ref")]
    extras    = [c for c in candidates if c.get("is_cross_ref")]

    if not extras:
        return candidates

    extras.sort(key=lambda c: c.get("source_score", 0.0), reverse=True)

    used_tokens = sum(
        reranker.count_tokens(c["payload"].get("text", c["payload"].get("chunk_text", "")))
        for c in originals
    )

    kept_extras = []
    for c in extras:
        text = c["payload"].get("text", c["payload"].get("chunk_text", ""))
        n_tokens = reranker.count_tokens(text)
        if used_tokens + n_tokens > max_tokens:
            continue
        used_tokens += n_tokens
        kept_extras.append(c)

    return originals + kept_extras


def _search_ho(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict],
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    use_cross_refs: bool = True,
    max_cross_ref_tokens: int | None = DEFAULT_MAX_CROSS_REF_TOKENS,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    HoRAG: law_kb_ho_fixedid에서 호/목/세목 단위 검색
    → (옵션) cross_refs로 관련 조항 확장 (무제한으로 일단 모음)
    → parent_chunk_id로 law_kb_jo_fixedid에서 조 전체 텍스트 fetch
      (최종 출력 단위는 항상 조 — 하위 단위는 검색 후보로만 쓰고 결과로는 안 남김)
    → (옵션) cross-ref로 추가된 후보만 토큰 예산 기준으로 트리밍
      (원본 후보는 항상 유지, parent fetch 이후 실제 조 텍스트 길이 기준으로 자름)
    → reranker
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_HO, fetch_k, alpha, rrf_k, cache)

    if use_cross_refs:
        candidates = _expand_with_cross_refs(candidates, client, cache)

    # parent fetch: 호/목/세목 → 조 텍스트로 교체 (최종 출력 단위 통일)
    candidates = _fetch_parent_texts(candidates, client, parent_collection=COLLECTION_JO, cache=cache)

    if use_cross_refs and max_cross_ref_tokens is not None:
        if reranker is not None:
            candidates = _trim_cross_refs_by_token_budget(candidates, reranker, max_cross_ref_tokens)
        else:
            print("  ⚠️  max_cross_ref_tokens가 설정됐지만 reranker가 없어 토큰 트리밍을 건너뜁니다.")

    if use_reranker and reranker and candidates:
        candidates = _rerank(clause_text, candidates, reranker, rerank_k, "ho", cache)

    return _build_law_refs(candidates, laws_ref, top_k, with_xref=use_cross_refs)


def review_contract_ho(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict] | None = None,
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    use_cross_refs: bool = True,
    max_cross_ref_tokens: int | None = DEFAULT_MAX_CROSS_REF_TOKENS,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """HoRAG 메인 인터페이스 (cross_refs 확장 포함, 별도 xref variant 없음)."""
    if laws_ref is None:
        laws_ref = load_laws_ref()

    clauses = chunk_contract(contract_text)
    results : list[ClauseResult] = []
    print(f"[HoRAG] 총 {len(clauses)}개 청크 검색 중...")

    for i, clause in enumerate(clauses, 1):
        print(f"  [{i}/{len(clauses)}] {clause['clause_number']} ...", end="\r")

        law_refs   = _search_ho(
            clause["clause_text"], client, model, laws_ref,
            reranker, use_reranker, use_cross_refs, max_cross_ref_tokens,
            top_k, alpha, fetch_k, rerank_k, rrf_k, cache,
        )
        categories = list(dict.fromkeys(r.category for r in law_refs if r.category))

        results.append(ClauseResult(
            clause_number=clause["clause_number"],
            clause_text  =clause["clause_text"],
            law_refs     =law_refs,
            categories   =categories,
        ))

    print(f"\n[HoRAG] ✅ 완료")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# JSON 변환
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def results_to_json(results: list[ClauseResult]) -> list[dict]:
    return [asdict(r) for r in results]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 편의: 단일 조항 검색 (sweep 스크립트에서 이 함수들을 직접 호출)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def search_jo(clause_text: str, client: QdrantClient, model: BGEM3FlagModel,
              laws_ref: dict, reranker=None, use_reranker=False,
              top_k=DEFAULT_TOP_K, alpha=DEFAULT_ALPHA, fetch_k=DEFAULT_FETCH_K,
              rerank_k=DEFAULT_RERANK_K, rrf_k=DEFAULT_RRF_K,
              cache: SweepCache | None = None) -> list[LawRef]:
    return _search_jo(clause_text, client, model, laws_ref, reranker,
                       use_reranker, top_k, alpha, fetch_k, rerank_k, rrf_k,
                       cache)


def search_ho(clause_text: str, client: QdrantClient, model: BGEM3FlagModel,
              laws_ref: dict, reranker=None, use_reranker=False, use_cross_refs=True,
              max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
              top_k=DEFAULT_TOP_K, alpha=DEFAULT_ALPHA, fetch_k=DEFAULT_FETCH_K,
              rerank_k=DEFAULT_RERANK_K, rrf_k=DEFAULT_RRF_K,
              cache: SweepCache | None = None) -> list[LawRef]:
    return _search_ho(clause_text, client, model, laws_ref, reranker,
                       use_reranker, use_cross_refs, max_cross_ref_tokens,
                       top_k, alpha, fetch_k, rerank_k, rrf_k, cache)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 프로덕션 인터페이스 — 확정된 조합(BEST_JO_CONFIG / BEST_HO_CONFIG)을 그대로
# 적용한 고수준 함수. 호출부에서 alpha/reranker on-off 조합을 매번 기억할
# 필요 없이 이 두 함수만 쓰면 된다.
#
#   review_contract()         : 메인 경로 — 계약서 전체를 조 단위로 검토.
#   get_detailed_citations()  : 보조 경로 — 계약서 조항 1개에 대해 호/목 단위
#                                 세부 근거가 필요할 때만 호출 (사용자가 "세부
#                                 근거 보기" 등을 요청했을 때 온디맨드로 사용).
#
# 주의: reranker가 하나로 통합됐으므로, 재sweep 후 BEST_JO_CONFIG/BEST_HO_CONFIG의
# fetch_k/rerank_k/rrf_k/max_cross_ref_tokens 값을 확정해서 채워 넣을 것.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def review_contract(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    reranker     : CrossEncoderReranker,
    laws_ref     : dict[str, dict] | None = None,
    top_k        : int = DEFAULT_TOP_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """메인 검토 경로. BEST_JO_CONFIG를 그대로 적용한 JoRAG 호출이다."""
    return review_contract_jo(
        contract_text, client, model, laws_ref,
        reranker=reranker,
        top_k=top_k, cache=cache,
        **BEST_JO_CONFIG,
    )


def get_detailed_citations(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    reranker     : CrossEncoderReranker,
    laws_ref     : dict[str, dict] | None = None,
    top_k        : int = DEFAULT_TOP_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    보조 경로. 계약서 조항 1개(review_contract가 반환한 ClauseResult.clause_text)에
    대해 호/목 단위 세부 근거가 필요할 때만 호출한다. BEST_HO_CONFIG를 그대로
    적용한 HoRAG 호출이다.

    review_contract()와 매 요청마다 같이 부르지 말 것 — 세부 근거는 사용자가
    명시적으로 요청했을 때만 온디맨드로 호출하는 게 설계 의도다 (하이브리드
    서비스 구조 — Notion 문서 참고).
    """
    if laws_ref is None:
        laws_ref = load_laws_ref()

    return search_ho(
        clause_text, client, model, laws_ref,
        reranker=reranker,
        top_k=top_k, cache=cache,
        **BEST_HO_CONFIG,
    )


Writing /content/yoonha_law_rag.py


In [5]:
%%writefile /content/yoonha_colab_upsert.py
"""
Workit - Colab용 로컬 Qdrant 셋업 + upsert 스크립트
파일명: rag/yoonha_colab_upsert.py

목적:
  Colab은 로컬 머신의 Qdrant(localhost:6333)에 접근할 수 없다. 대신 이미
  임베딩이 끝난 4개 파일(vectors_*.npz, sparse_weights_*.json)과 payload
  메타데이터(chunks_*_fixedid.json)를 Colab에 직접 업로드해서, Colab
  안에서 "로컬 파일 기반" Qdrant를 새로 띄우고 그 안에 upsert한다.
  서버 프로세스가 따로 필요 없다 (QdrantClient(path=...) 모드).

  이렇게 하면:
    - ngrok/터널링 없이 Colab 세션 안에서 완결됨
    - 로컬 머신을 계속 켜둘 필요 없음
    - GPU는 reranker/embedding 계산에만 쓰이고, Qdrant 자체는 CPU로 충분

전제:
  - vectors_jo_fixedid.npz / vectors_ho_fixedid.npz : {"vectors": (N,1024) float16,
    "chunk_ids": (N,) 문자열 배열}
  - sparse_weights_jo_fixedid.json / sparse_weights_ho_fixedid.json : 길이 N인
    리스트, vectors의 chunk_ids와 "같은 순서"로 정렬돼 있다고 이미 확인함
    (jo/ho 둘 다 순서 일치, 중복 0 — 실제 업로드 파일로 검증 완료).
  - chunks_jo_fixedid.json / chunks_ho_fixedid.json : chunk_id별 payload
    메타데이터(text, parent_chunk_id, cross_refs, law_name, article_number,
    title, hierarchy, is_ref_article, is_upper_law 등). vectors의 chunk_ids와
    "순서까지" 동일하다고 확인했지만, 이 스크립트는 순서에 의존하지 않고
    chunk_id로 다시 매핑해서 병합한다 (더 안전한 방식).

  주의 — 지금 chunks_*_fixedid.json에는 category, is_risk_ref 필드가 없다.
  yoonha_law_rag.py의 _build_law_refs가 이 필드들을 payload.get(key, 기본값)
  으로 읽기 때문에 없어도 에러는 안 나고 그냥 빈 문자열/False로 채워진다.
  나중에 laws_ref.json(load_laws_ref)에서 category를 따로 채워주는 구조라
  RAG 파이프라인 동작 자체엔 문제 없음.

사용법 (Colab 셀에서):
    !pip install qdrant-client --quiet
    !python yoonha_colab_upsert.py
  또는 노트북 안에서 build_local_qdrant() 함수를 직접 호출해도 된다.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    SparseVectorParams,
    PointStruct,
    SparseVector,
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 경로 — Colab에서 업로드한 파일 위치에 맞게 수정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA_DIR = Path("/content")  # 업로드한 4+2개 파일이 있는 디렉터리

VECTORS_JO      = DATA_DIR / "vectors_jo_fixedid.npz"
VECTORS_HO      = DATA_DIR / "vectors_ho_fixedid.npz"
SPARSE_JO       = DATA_DIR / "sparse_weights_jo_fixedid.json"
SPARSE_HO       = DATA_DIR / "sparse_weights_ho_fixedid.json"
CHUNKS_JO       = DATA_DIR / "chunks_jo_fixedid.json"
CHUNKS_HO       = DATA_DIR / "chunks_ho_fixedid.json"

# Colab 세션 안에 로컬로 만들 Qdrant 저장 경로 (서버 프로세스 없이 파일 기반)
QDRANT_LOCAL_PATH = "/content/qdrant_local"

COLLECTION_JO = "law_kb_jo_fixedid"
COLLECTION_HO = "law_kb_ho_fixedid"

DENSE_DIM = 1024  # BGE-M3 dense 차원 (npz vectors.shape[1]로 실제 검증도 함께 함)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 로드 / 병합
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _load_merged(vectors_path: Path, sparse_path: Path, chunks_path: Path) -> list[dict]:
    """
    vectors(.npz) + sparse_weights(.json) + chunks(.json, payload)를
    chunk_id 기준으로 병합해서 [{chunk_id, dense_vec, sparse_vec, payload}, ...] 반환.
    """
    npz = np.load(vectors_path, allow_pickle=True)
    dense_vectors = npz["vectors"]          # (N, 1024) float16
    chunk_ids     = npz["chunk_ids"].tolist()

    assert dense_vectors.shape[1] == DENSE_DIM, (
        f"예상한 dense 차원({DENSE_DIM})과 실제({dense_vectors.shape[1]})가 다릅니다 — "
        f"임베딩 모델이 바뀐 건 아닌지 확인하세요."
    )

    with open(sparse_path, encoding="utf-8") as f:
        sparse_weights = json.load(f)  # chunk_ids와 같은 순서의 리스트

    assert len(sparse_weights) == len(chunk_ids), (
        f"sparse_weights 개수({len(sparse_weights)})와 chunk_ids 개수({len(chunk_ids)})가 다릅니다."
    )

    with open(chunks_path, encoding="utf-8") as f:
        chunks_list = json.load(f)

    payload_by_id = {c["chunk_id"]: c for c in chunks_list}

    missing = [cid for cid in chunk_ids if cid not in payload_by_id]
    if missing:
        print(f"  ⚠️  payload를 못 찾은 chunk_id {len(missing)}개 (예: {missing[:5]}) "
              f"— 해당 chunk는 text 없이 upsert됩니다.")

    merged = []
    for i, cid in enumerate(chunk_ids):
        payload = dict(payload_by_id.get(cid, {}))
        payload["chunk_id"] = cid  # payload 안에도 chunk_id를 넣어야 검색 결과에서 식별 가능
        merged.append({
            "chunk_id":   cid,
            "dense_vec":  dense_vectors[i].astype(np.float32).tolist(),
            "sparse_vec": sparse_weights[i],  # {"token_id": weight, ...}
            "payload":    payload,
        })

    return merged


def _to_qdrant_points(merged: list[dict]) -> list[PointStruct]:
    points = []
    for item in merged:
        sparse = item["sparse_vec"]
        indices = [int(k) for k in sparse.keys()]
        values  = [float(v) for v in sparse.values()]

        points.append(PointStruct(
            id=abs(hash(item["chunk_id"])) % (2**63),  # Qdrant point id는 int/UUID만 허용, chunk_id는 payload에 문자열로 그대로 보존
            vector={
                "dense":  item["dense_vec"],
                "sparse": SparseVector(indices=indices, values=values),
            },
            payload=item["payload"],
        ))
    return points


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 컬렉션 생성 + upsert
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _create_collection(client: QdrantClient, name: str) -> None:
    if client.collection_exists(name):
        print(f"  ℹ️  {name} 이미 존재 — 삭제 후 재생성")
        client.delete_collection(name)

    client.create_collection(
        collection_name=name,
        vectors_config={
            "dense": VectorParams(size=DENSE_DIM, distance=Distance.COSINE),
        },
        sparse_vectors_config={
            "sparse": SparseVectorParams(),
        },
    )


def _upsert_batched(client: QdrantClient, name: str, points: list[PointStruct], batch_size: int = 256) -> None:
    total = len(points)
    for i in range(0, total, batch_size):
        batch = points[i : i + batch_size]
        client.upsert(collection_name=name, points=batch)
        print(f"  [{name}] {min(i + batch_size, total)}/{total} upsert 완료", end="\r")
    print()


def build_local_qdrant(recreate: bool = True) -> QdrantClient:
    """
    Colab 세션 안에서 로컬(파일 기반) Qdrant를 만들고, 업로드된 4+2개 파일로
    law_kb_jo_fixedid / law_kb_ho_fixedid 컬렉션을 채운다.
    반환된 client를 그대로 yoonha_rag_eval*.py의 QdrantClient 자리에 넣으면 된다.
    """
    client = QdrantClient(path=QDRANT_LOCAL_PATH)

    print("📦 jo 데이터 로드/병합 중...")
    merged_jo = _load_merged(VECTORS_JO, SPARSE_JO, CHUNKS_JO)
    print(f"  -> {len(merged_jo)}개 chunk 병합 완료")

    print("📦 ho 데이터 로드/병합 중...")
    merged_ho = _load_merged(VECTORS_HO, SPARSE_HO, CHUNKS_HO)
    print(f"  -> {len(merged_ho)}개 chunk 병합 완료")

    print(f"🗂️  {COLLECTION_JO} 컬렉션 생성...")
    _create_collection(client, COLLECTION_JO)
    _upsert_batched(client, COLLECTION_JO, _to_qdrant_points(merged_jo))

    print(f"🗂️  {COLLECTION_HO} 컬렉션 생성...")
    _create_collection(client, COLLECTION_HO)
    _upsert_batched(client, COLLECTION_HO, _to_qdrant_points(merged_ho))

    jo_count = client.count(COLLECTION_JO).count
    ho_count = client.count(COLLECTION_HO).count
    print(f"\n✅ 완료 — {COLLECTION_JO}: {jo_count}개, {COLLECTION_HO}: {ho_count}개")

    return client


if __name__ == "__main__":
    build_local_qdrant()



Writing /content/yoonha_colab_upsert.py


## 3. 업로드 파일 존재 확인

In [8]:
from pathlib import Path

REQUIRED_FILES = [
    "/content/vectors_jo_fixedid.npz",
    "/content/vectors_ho_fixedid.npz",
    "/content/sparse_weights_jo_fixedid.json",
    "/content/sparse_weights_ho_fixedid.json",
    "/content/chunks_jo_fixedid.json",
    "/content/chunks_ho_fixedid.json",
    "/content/eval_set.json",  # 50개/100개 등 파일명이 다르면 여기만 바꾸세요
]

missing = [f for f in REQUIRED_FILES if not Path(f).exists()]
if missing:
    print("⚠️  아직 안 올라온 파일:")
    for f in missing:
        print("  -", f)
    print("\n왼쪽 파일 탐색기(📁)에서 위 파일들을 /content/에 업로드한 뒤 이 셀을 다시 실행하세요.")
else:
    print("✅ 필요한 파일이 모두 있습니다.")

✅ 필요한 파일이 모두 있습니다.


## 4. 로컬 Qdrant 생성 + upsert

`/content/qdrant_local`에 파일 기반 Qdrant를 만들고,
`law_kb_jo_fixedid`, `law_kb_ho_fixedid` 컬렉션을 채웁니다.
서버 프로세스가 따로 필요 없습니다.

In [27]:
import sys
sys.path.insert(0, "/content")

from yoonha_colab_upsert import build_local_qdrant

client = build_local_qdrant()

# 로컬(파일 기반) Qdrant는 경로 하나에 클라이언트가 동시에 하나만 붙을 수 있음.
# 아래 sweep 셀은 !python으로 "별도 프로세스"에서 같은 경로를 다시 여니까,
# 여기서 만든 client를 계속 열어두면 그쪽에서 파일 락 충돌(AlreadyLocked)이 남.
# upsert가 끝났으면 바로 닫아서 락을 풀어준다.
client.close()
print("✅ upsert 완료, client 닫음 (sweep 프로세스가 같은 경로에 새로 붙을 수 있게)")

📦 jo 데이터 로드/병합 중...
  -> 1186개 chunk 병합 완료
📦 ho 데이터 로드/병합 중...
  -> 7640개 chunk 병합 완료
🗂️  law_kb_jo_fixedid 컬렉션 생성...
  ℹ️  law_kb_jo_fixedid 이미 존재 — 삭제 후 재생성
  [law_kb_jo_fixedid] 1186/1186 upsert 완료
🗂️  law_kb_ho_fixedid 컬렉션 생성...
  ℹ️  law_kb_ho_fixedid 이미 존재 — 삭제 후 재생성
  [law_kb_ho_fixedid] 7640/7640 upsert 완료

✅ 완료 — law_kb_jo_fixedid: 1186개, law_kb_ho_fixedid: 7640개
✅ upsert 완료, client 닫음 (sweep 프로세스가 같은 경로에 새로 붙을 수 있게)


## 5. 단계적(Staged) sweep 실행

`QDRANT_LOCAL_PATH`, `RERANKER_DEVICE` 환경변수만 잡아주면
`yoonha_rag_eval_staged.py` 안의 client/reranker 생성 로직이 자동으로
Colab용(로컬 파일 기반 Qdrant + GPU)으로 분기합니다.

단계 구성:
- 0단계: reranker OFF baseline
- 1단계: alpha × rrf_k (20개)
- 2단계: fetch_k × rerank_k (rerank_k ≤ fetch_k만, 최대 10개)
- 3단계 (HoRAG만): use_cross_refs × max_cross_ref_tokens (5개)

JoRAG ≈ 31개, HoRAG ≈ 36개 — 기존 완전 교차곱(380/760개) 대비 훨씬 적습니다.

In [28]:
%%writefile /content/yoonha_rag_eval_staged.py
"""
Workit - RAG 파라미터 단계적(staged) sweep, reranker 단일화 버전
파일명: rag/yoonha_rag_eval_staged.py

완전 교차곱(alpha × rrf_k × fetch_k × rerank_k × ...)을 한 번에 다 돌리면
JoRAG 380개, HoRAG 760개까지 불어나서(2280개까지도 감), 대신 아래처럼
단계적으로 좁혀서 돈다.

  0단계 (baseline)  : reranker 아예 끈 raw hybrid search 성능 확인
                       (reranker가 실제로 도움이 되는지 자체를 검증하는 참고용,
                        alpha/rrf_k는 기본값 고정)
  1단계 (alpha×rrf_k): fetch_k/rerank_k는 기본값(DEFAULT_FETCH_K/DEFAULT_RERANK_K)에
                       고정하고, alpha(5) × rrf_k(4) = 20개만 sweep.
                       이 둘은 raw_search 캐시를 안 건드리는 "거의 공짜" 축이라
                       비용 부담 없이 넓게 봐도 된다.
  2단계 (fetch_k×rerank_k): 1단계에서 찾은 최적 alpha/rrf_k를 고정하고,
                       fetch_k(4) × rerank_k(4, rerank_k<=fetch_k 필터) 를 sweep.
                       fetch_k는 raw_search 재계산이 발생하는 비싼 축이라
                       여기서만 별도로 좁혀서 본다.
  3단계 (HoRAG 전용, cross-ref): 1·2단계 최적값을 모두 고정하고,
                       use_cross_refs on/off + max_cross_ref_tokens(4개)를 sweep.
                       use_cross_refs=False일 때는 max_cross_ref_tokens을
                       대표값(None)으로 collapse해서 중복 조합을 만들지 않는다.

이 구조로 돌면:
  JoRAG : 1(baseline) + 20(1단계) + ~15(2단계, rerank_k<=fetch_k 필터 후) = 36개
  HoRAG : 위 + 3단계(use_cross_refs=False 1개 + True일 때 4개 = 5개) = 41개

reranker는 bge-reranker-v2-m3 하나로 고정 (yoonha_law_rag.py 개정판 기준).

  Jo는 LLM에 top-2~3개만 넘기는 용도라 최적 조합 선정 기준을
  Recall@3 (동률 시 MRR)으로 잡는다. Ho는 top-10까지 넘기므로
  기존처럼 MRR 기준을 유지한다. get_best_row(df, sort_col=...)로
  단계별로 다르게 지정한다.

  Recall@k는 gt_docs 중 "하나라도" top-k에 걸리면 hit으로 치는 지표라
  멀티독(정답이 여러 개인 쿼리) 케이스에서 "top-k 안에 정답이 다 담기는지"는
  못 잡아낸다. 이를 보완하기 위해 FullCoverage@k(정답 전체가 top-k에
  subset으로 포함되는 비율)를 함께 계산한다. Jo처럼 top-k를 작게
  자를수록 이 지표가 실질적인 "노이즈 없는 완전한 근거 제공" 여부를 보여준다.

출력:
  - eval_stage0_baseline.csv
  - eval_stage1_alpha_rrf.csv
  - eval_stage2_fetchk_rerankk.csv
  - eval_stage3_crossref.csv        (HoRAG만)
  - eval_final_best.csv             (variant별 최종 확정 조합 1줄씩)

사용법:
    pip install qdrant-client FlagEmbedding transformers torch pandas
    python rag/yoonha_rag_eval_staged.py
"""

from __future__ import annotations

import itertools
import json
import os
import time
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient

from yoonha_law_rag import (
    load_embed_model,
    load_laws_ref,
    load_reranker,
    search_jo,
    search_ho,
    SweepCache,
    QDRANT_HOST,
    QDRANT_PORT,
    DEFAULT_FETCH_K,
    DEFAULT_RERANK_K,
    DEFAULT_RRF_K,
    DEFAULT_ALPHA,
    DEFAULT_MAX_CROSS_REF_TOKENS,
)

# Colab 등 로컬 서버(localhost:6333)에 못 붙는 환경에서는
# QDRANT_LOCAL_PATH 환경변수를 잡아주면 파일 기반 로컬 Qdrant를 쓴다.
# GPU 리랭커를 쓰려면 RERANKER_DEVICE=cuda로 잡아준다 (기본값 cpu).
QDRANT_LOCAL_PATH_ENV = os.environ.get("QDRANT_LOCAL_PATH")
RERANKER_DEVICE = os.environ.get("RERANKER_DEVICE", "cpu")


def get_qdrant_client() -> QdrantClient:
    if QDRANT_LOCAL_PATH_ENV:
        print(f"🗄️  로컬(파일 기반) Qdrant 사용: {QDRANT_LOCAL_PATH_ENV}")
        return QdrantClient(path=QDRANT_LOCAL_PATH_ENV)
    print(f"🗄️  서버 Qdrant 사용: {QDRANT_HOST}:{QDRANT_PORT}")
    return QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)


GOLD_STANDARD_PATH = Path("/content/eval_set.json")  # 50개든 100개든 경로만 이거면 됨
OUT_DIR = Path("/content")

GT_FIELD_BY_VARIANT = {
    "jo": "relevant_docs_jo",
    "ho": "relevant_docs_ho",
}

# 최종 recall을 볼 top_k 목록 — 한 번 검색한 결과에서 전부 잘라 계산 (재검색 없음)
# 2를 추가한 이유: Jo가 top-2~3만 LLM에 넘기는 용도라 top-2 단독 성능도 봐야 함
RECALL_AT_KS = [1, 2, 3, 5, 10]
TOP_K_EVAL   = max(RECALL_AT_KS)

# Jo/Ho 각각 최적 조합을 고를 때 기준으로 삼을 컬럼.
# Jo: top-2~3만 넘기므로 그 컷오프 안에서의 recall을 직접 최적화.
# Ho: top-10까지 넘기므로 기존처럼 MRR(순위 전체의 정밀도) 기준 유지.
JO_SORT_COL = "Recall@3"
HO_SORT_COL = "MRR"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 각 단계 sweep 그리드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ALPHA_GRID    = [0.3, 0.5, 0.6, 0.7, 1.0]   # 1.0 = dense only
RRF_K_GRID    = [20, 40, 60, 100]           # 60 = 기존 하드코딩값
FETCH_K_GRID  = [20, 30, 50, 80]
RERANK_K_GRID = [5, 10, 20, 30]

USE_CROSS_REFS_GRID = [True, False]
MAX_CROSS_REF_TOKENS_GRID = [500, 1000, 2000, None]  # None = 트리밍 없음(무제한)


def load_gold_standard(path: Path = GOLD_STANDARD_PATH) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 공통 평가 함수 — variant(jo/ho)에 따라 search_jo/search_ho만 다르게 호출
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def evaluate_params(
    variant      : str,
    client       : QdrantClient,
    model,
    laws_ref     : dict,
    reranker,
    gold_standard: list[dict],
    cache        : SweepCache,
    params       : dict,
) -> dict:
    """
    params는 search_jo/search_ho에 그대로 넘길 키워드 인자 dict
    (alpha, rrf_k, fetch_k, rerank_k, use_reranker, [use_cross_refs, max_cross_ref_tokens]).

    Recall@k       : gt_docs 중 "하나라도" top-k 안에 있으면 hit
    FullCoverage@k : gt_docs "전부"가 top-k 안에 포함되면 hit
                     (멀티독 쿼리에서 top-k를 작게 자를 때 정답 누락 여부를 잡아냄)
    """
    recall_hits = {k: 0 for k in RECALL_AT_KS}
    full_coverage_hits = {k: 0 for k in RECALL_AT_KS}
    mrr = 0.0
    n = len(gold_standard)
    t0 = time.time()
    gt_field = GT_FIELD_BY_VARIANT[variant]

    search_fn = search_jo if variant == "jo" else search_ho

    for item in gold_standard:
        gt_docs = set(item[gt_field])

        law_refs = search_fn(
            clause_text=item["query"],
            client=client,
            model=model,
            laws_ref=laws_ref,
            reranker=reranker,
            top_k=TOP_K_EVAL,
            cache=cache,
            **params,
        )
        ranked_ids = [r.chunk_id for r in law_refs]

        for k in RECALL_AT_KS:
            top_k_set = set(ranked_ids[:k])
            if gt_docs & top_k_set:
                recall_hits[k] += 1
            if gt_docs and gt_docs.issubset(top_k_set):
                full_coverage_hits[k] += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break

    elapsed = time.time() - t0

    result = {"variant": variant, **params}
    for k in RECALL_AT_KS:
        result[f"Recall@{k}"] = round(recall_hits[k] / n, 4)
        result[f"FullCoverage@{k}"] = round(full_coverage_hits[k] / n, 4)
    result["MRR"] = round(mrr / n, 4)
    result["avg_sec_per_query"] = round(elapsed / n, 3)
    return result


def run_grid(
    variant, client, model, laws_ref, reranker, gold_standard, cache,
    fixed_params: dict, grid_params: dict, stage_name: str,
) -> pd.DataFrame:
    """
    fixed_params: 이번 단계에서 고정할 값들
    grid_params : {key: [values...]} 형태, itertools.product로 전부 조합
    """
    keys = list(grid_params.keys())
    value_lists = [grid_params[k] for k in keys]
    combos = list(itertools.product(*value_lists))

    print(f"\n━━━ [{stage_name}] {variant} — {len(combos)}개 조합 ━━━")

    rows = []
    for i, combo_values in enumerate(combos, 1):
        params = {**fixed_params, **dict(zip(keys, combo_values))}
        print(f"  [{i}/{len(combos)}] {params} ...", end="")
        result = evaluate_params(variant, client, model, laws_ref, reranker, gold_standard, cache, params)
        rows.append(result)
        if variant == "jo":
            print(f" -> MRR={result['MRR']} Recall@2={result['Recall@2']} Recall@3={result['Recall@3']} "
                  f"FullCoverage@3={result['FullCoverage@3']} ({result['avg_sec_per_query']}초/쿼리)")
        else:
            print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} ({result['avg_sec_per_query']}초/쿼리)")

    df = pd.DataFrame(rows).sort_values("MRR", ascending=False)
    return df


def get_best_row(df: pd.DataFrame, sort_col: str = "MRR") -> dict:
    return df.sort_values(sort_col, ascending=False).iloc[0].to_dict()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# JoRAG 단계적 sweep
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def sweep_jo(client, model, laws_ref, reranker, gold_standard, cache) -> dict:
    # 0단계: baseline (reranker 끈 raw hybrid search)
    baseline_params = dict(
        alpha=DEFAULT_ALPHA, rrf_k=DEFAULT_RRF_K,
        fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K,
        use_reranker=False,
    )
    baseline = evaluate_params("jo", client, model, laws_ref, reranker, gold_standard, cache, baseline_params)
    print(f"\n[jo baseline, reranker OFF] MRR={baseline['MRR']} Recall@3={baseline['Recall@3']} "
          f"FullCoverage@3={baseline['FullCoverage@3']}")

    # 1단계: alpha × rrf_k (fetch_k/rerank_k 기본값 고정, reranker ON)
    df_stage1 = run_grid(
        "jo", client, model, laws_ref, reranker, gold_standard, cache,
        fixed_params=dict(fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K, use_reranker=True),
        grid_params=dict(alpha=ALPHA_GRID, rrf_k=RRF_K_GRID),
        stage_name="1단계 alpha×rrf_k",
    )
    df_stage1.to_csv(OUT_DIR / "eval_stage1_alpha_rrf_jo.csv", index=False)
    best1 = get_best_row(df_stage1, sort_col=JO_SORT_COL)
    print(f"[jo 1단계 최적 ({JO_SORT_COL} 기준)] alpha={best1['alpha']}, rrf_k={best1['rrf_k']} "
          f"-> {JO_SORT_COL}={best1[JO_SORT_COL]} MRR={best1['MRR']}")

    # 2단계: fetch_k × rerank_k (1단계 최적 alpha/rrf_k 고정)
    combos_fk_rk = [
        (fk, rk) for fk, rk in itertools.product(FETCH_K_GRID, RERANK_K_GRID) if rk <= fk
    ]
    df_stage2_rows = []
    print(f"\n━━━ [2단계 fetch_k×rerank_k] jo — {len(combos_fk_rk)}개 조합 ━━━")
    for i, (fk, rk) in enumerate(combos_fk_rk, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=fk, rerank_k=rk, use_reranker=True,
        )
        print(f"  [{i}/{len(combos_fk_rk)}] fetch_k={fk} rerank_k={rk} ...", end="")
        result = evaluate_params("jo", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage2_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@2={result['Recall@2']} Recall@3={result['Recall@3']} "
              f"FullCoverage@3={result['FullCoverage@3']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage2 = pd.DataFrame(df_stage2_rows).sort_values("MRR", ascending=False)
    df_stage2.to_csv(OUT_DIR / "eval_stage2_fetchk_rerankk_jo.csv", index=False)
    best2 = get_best_row(df_stage2, sort_col=JO_SORT_COL)
    print(f"[jo 2단계 최적 ({JO_SORT_COL} 기준)] fetch_k={best2['fetch_k']}, rerank_k={best2['rerank_k']} "
          f"-> {JO_SORT_COL}={best2[JO_SORT_COL]} MRR={best2['MRR']}")

    final = dict(
        variant="jo",
        alpha=best1["alpha"], rrf_k=best1["rrf_k"],
        fetch_k=best2["fetch_k"], rerank_k=best2["rerank_k"],
        use_reranker=True,
        MRR=best2["MRR"],
        **{f"Recall@{k}": best2[f"Recall@{k}"] for k in RECALL_AT_KS},
        **{f"FullCoverage@{k}": best2[f"FullCoverage@{k}"] for k in RECALL_AT_KS},
    )
    return final


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HoRAG 단계적 sweep (1·2단계는 jo와 동일 구조 + 3단계 cross-ref 추가)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def sweep_ho(client, model, laws_ref, reranker, gold_standard, cache) -> dict:
    baseline_params = dict(
        alpha=DEFAULT_ALPHA, rrf_k=DEFAULT_RRF_K,
        fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K,
        use_reranker=False, use_cross_refs=True,
        max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
    )
    baseline = evaluate_params("ho", client, model, laws_ref, reranker, gold_standard, cache, baseline_params)
    print(f"\n[ho baseline, reranker OFF] MRR={baseline['MRR']} Recall@10={baseline['Recall@10']}")

    # 1단계: alpha × rrf_k (fetch_k/rerank_k 기본값, cross_refs 기본 ON, 트리밍 없음)
    df_stage1 = run_grid(
        "ho", client, model, laws_ref, reranker, gold_standard, cache,
        fixed_params=dict(
            fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K, use_reranker=True,
            use_cross_refs=True, max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
        ),
        grid_params=dict(alpha=ALPHA_GRID, rrf_k=RRF_K_GRID),
        stage_name="1단계 alpha×rrf_k",
    )
    df_stage1.to_csv(OUT_DIR / "eval_stage1_alpha_rrf_ho.csv", index=False)
    best1 = get_best_row(df_stage1, sort_col=HO_SORT_COL)
    print(f"[ho 1단계 최적] alpha={best1['alpha']}, rrf_k={best1['rrf_k']} -> MRR={best1['MRR']}")

    # 2단계: fetch_k × rerank_k
    combos_fk_rk = [
        (fk, rk) for fk, rk in itertools.product(FETCH_K_GRID, RERANK_K_GRID) if rk <= fk
    ]
    df_stage2_rows = []
    print(f"\n━━━ [2단계 fetch_k×rerank_k] ho — {len(combos_fk_rk)}개 조합 ━━━")
    for i, (fk, rk) in enumerate(combos_fk_rk, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=fk, rerank_k=rk, use_reranker=True,
            use_cross_refs=True, max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
        )
        print(f"  [{i}/{len(combos_fk_rk)}] fetch_k={fk} rerank_k={rk} ...", end="")
        result = evaluate_params("ho", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage2_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage2 = pd.DataFrame(df_stage2_rows).sort_values("MRR", ascending=False)
    df_stage2.to_csv(OUT_DIR / "eval_stage2_fetchk_rerankk_ho.csv", index=False)
    best2 = get_best_row(df_stage2, sort_col=HO_SORT_COL)
    print(f"[ho 2단계 최적] fetch_k={best2['fetch_k']}, rerank_k={best2['rerank_k']} -> MRR={best2['MRR']}")

    # 3단계 (ho 전용): use_cross_refs on/off + max_cross_ref_tokens
    #   use_cross_refs=False면 max_cross_ref_tokens는 의미 없으므로 대표값(None) 하나로만.
    combos_stage3 = [(False, None)] + [(True, t) for t in MAX_CROSS_REF_TOKENS_GRID]
    df_stage3_rows = []
    print(f"\n━━━ [3단계 use_cross_refs×max_cross_ref_tokens] ho — {len(combos_stage3)}개 조합 ━━━")
    for i, (use_xref, max_tok) in enumerate(combos_stage3, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=best2["fetch_k"], rerank_k=best2["rerank_k"], use_reranker=True,
            use_cross_refs=use_xref, max_cross_ref_tokens=max_tok,
        )
        print(f"  [{i}/{len(combos_stage3)}] use_cross_refs={use_xref} max_cross_ref_tokens={max_tok} ...", end="")
        result = evaluate_params("ho", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage3_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage3 = pd.DataFrame(df_stage3_rows).sort_values("MRR", ascending=False)
    df_stage3.to_csv(OUT_DIR / "eval_stage3_crossref_ho.csv", index=False)
    best3 = get_best_row(df_stage3, sort_col=HO_SORT_COL)
    print(f"[ho 3단계 최적] use_cross_refs={best3['use_cross_refs']}, "
          f"max_cross_ref_tokens={best3['max_cross_ref_tokens']} -> MRR={best3['MRR']}")

    final = dict(
        variant="ho",
        alpha=best1["alpha"], rrf_k=best1["rrf_k"],
        fetch_k=best2["fetch_k"], rerank_k=best2["rerank_k"],
        use_reranker=True,
        use_cross_refs=best3["use_cross_refs"], max_cross_ref_tokens=best3["max_cross_ref_tokens"],
        MRR=best3["MRR"],
        **{f"Recall@{k}": best3[f"Recall@{k}"] for k in RECALL_AT_KS},
        **{f"FullCoverage@{k}": best3[f"FullCoverage@{k}"] for k in RECALL_AT_KS},
    )
    return final


def main():
    client   = get_qdrant_client()
    model    = load_embed_model()
    laws_ref = load_laws_ref()
    gold_standard = load_gold_standard()

    print(f"실버 스탠다드 {len(gold_standard)}개 로드 완료")

    # reranker는 무거우니 한 번만 로드 (grid 안에서 껐다 켰다는 플래그로만 토글)
    reranker = load_reranker(device=RERANKER_DEVICE)

    # 전체 단계에서 재사용할 캐시 — jo/ho 두 variant, 모든 단계에서 공유
    cache = SweepCache()

    t_start = time.time()

    print("\n" + "=" * 60)
    print("JoRAG 단계적 sweep 시작")
    print("=" * 60)
    final_jo = sweep_jo(client, model, laws_ref, reranker, gold_standard, cache)

    print("\n" + "=" * 60)
    print("HoRAG 단계적 sweep 시작")
    print("=" * 60)
    final_ho = sweep_ho(client, model, laws_ref, reranker, gold_standard, cache)

    t_total = time.time() - t_start

    df_final = pd.DataFrame([final_jo, final_ho])
    df_final.to_csv(OUT_DIR / "eval_final_best.csv", index=False)

    print("\n" + "=" * 60)
    print("✅ 전체 단계적 sweep 완료")
    print("=" * 60)
    print(f"⏱️  총 소요 시간: {t_total:.1f}초 ({t_total/60:.1f}분)")
    print(f"📦 최종 캐시 통계: {cache.stats()}")
    print("\n=== 최종 확정 조합 ===")
    print(df_final.to_string(index=False))


if __name__ == "__main__":
    main()

Overwriting /content/yoonha_rag_eval_staged.py


In [29]:
!nvidia-smi

Sun Jul  5 14:08:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [30]:
import os

os.environ["QDRANT_LOCAL_PATH"] = "/content/qdrant_local"
os.environ["RERANKER_DEVICE"] = "cuda"

!python /content/yoonha_rag_eval_staged.py

🗄️  로컬(파일 기반) Qdrant 사용: /content/qdrant_local
📦 임베딩 모델 로드: BAAI/bge-m3
Fetching 30 files: 100% 30/30 [00:00<00:00, 6167.19it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 391/391 [00:08<00:00, 47.31it/s]
  ⚠️  laws_ref.json 없음: /data/hn_seed/law_refs.json
실버 스탠다드 100개 로드 완료
📦 Re-ranker 로드: BAAI/bge-reranker-v2-m3
Loading weights: 100% 393/393 [00:00<00:00, 1489.76it/s]

JoRAG 단계적 sweep 시작

[jo baseline, reranker OFF] MRR=0.6583 Recall@3=0.81 FullCoverage@3=0.67

━━━ [1단계 alpha×rrf_k] jo — 20개 조합 ━━━
  [1/20] {'fetch_k': 50, 'rerank_k': 10, 'use_reranker': True, 'alpha': 0.3, 'rrf_k': 20} ... -> MRR=0.8837 Recall@2=0.95 Recall@3=0.97 FullCoverage@3=0.83 (6.431초/쿼리)
  [2/20] {'fetch_k': 50, 'rerank_k': 10, 'use_reranker': True, 'alpha': 0.3, 'rrf_k': 40} ... -> MRR=0.8837 Recall@2=0.95 Recall@3=0.97 FullCoverage@3=0.83 (0.0초/쿼리)
  [3/20] {'fetch_k': 50, 'rerank_k': 10, 'use_reranker': True, 'alpha': 0.3, 'rrf_k': 60} ... -> MRR=0.8837 Recall@2=0.95 Recall@3=0.97 Full

## 6. 결과 확인 + 다운로드

In [31]:
import pandas as pd

df_final = pd.read_csv("/content/eval_final_best.csv")
print("=== 최종 확정 조합 (jo/ho) ===")
df_final

=== 최종 확정 조합 (jo/ho) ===


,variant,alpha,rrf_k,fetch_k,rerank_k,use_reranker,MRR,Recall@1,Recall@2,Recall@3,Recall@5,Recall@10,FullCoverage@1,FullCoverage@2,FullCoverage@3,FullCoverage@5,FullCoverage@10,use_cross_refs,max_cross_ref_tokens
0,jo,0.3,20,80,20,True,0.8837,0.80,0.95,0.97,0.98,0.98,0.59,0.77,0.83,0.84,0.89,NaN,NaN
1,ho,0.3,20,80,10,True,0.7065,0.59,0.75,0.77,0.84,0.93,0.44,0.54,0.55,0.61,0.74,True,NaN


In [32]:
# 각 단계별 상세 결과도 확인 가능
df_stage1_jo = pd.read_csv("/content/eval_stage1_alpha_rrf_jo.csv")
df_stage2_jo = pd.read_csv("/content/eval_stage2_fetchk_rerankk_jo.csv")
df_stage1_ho = pd.read_csv("/content/eval_stage1_alpha_rrf_ho.csv")
df_stage2_ho = pd.read_csv("/content/eval_stage2_fetchk_rerankk_ho.csv")
df_stage3_ho = pd.read_csv("/content/eval_stage3_crossref_ho.csv")

print("=== jo 1단계 (alpha×rrf_k) 상위 5개 ===")
display(df_stage1_jo.head())

print("=== jo 2단계 (fetch_k×rerank_k) 상위 5개 ===")
display(df_stage2_jo.head())

print("=== ho 1단계 (alpha×rrf_k) 상위 5개 ===")
display(df_stage1_ho.head())

print("=== ho 2단계 (fetch_k×rerank_k) 상위 5개 ===")
display(df_stage2_ho.head())

print("=== ho 3단계 (use_cross_refs×max_cross_ref_tokens) 상위 5개 ===")
display(df_stage3_ho.head())

=== jo 1단계 (alpha×rrf_k) 상위 5개 ===


,variant,fetch_k,rerank_k,use_reranker,alpha,rrf_k,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,jo,50,10,True,0.3,20,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,6.431
1,jo,50,10,True,0.3,40,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
2,jo,50,10,True,0.3,60,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
3,jo,50,10,True,0.3,100,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
4,jo,50,10,True,0.5,20,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000


=== jo 2단계 (fetch_k×rerank_k) 상위 5개 ===


,variant,alpha,rrf_k,fetch_k,rerank_k,use_reranker,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,jo,0.3,20,80,20,True,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
1,jo,0.3,20,80,10,True,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
2,jo,0.3,20,80,5,True,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.84,0.8837,3.375
3,jo,0.3,20,50,30,True,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
4,jo,0.3,20,50,20,True,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000


=== ho 1단계 (alpha×rrf_k) 상위 5개 ===


,variant,fetch_k,rerank_k,use_reranker,use_cross_refs,max_cross_ref_tokens,alpha,rrf_k,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,ho,50,10,True,True,NaN,0.3,20,0.59,0.44,0.74,0.54,0.76,0.55,0.84,0.62,0.94,0.76,0.7041,13.152
1,ho,50,10,True,True,NaN,0.3,40,0.59,0.44,0.74,0.54,0.76,0.55,0.84,0.62,0.94,0.76,0.7041,0.272
2,ho,50,10,True,True,NaN,0.3,60,0.59,0.44,0.74,0.54,0.76,0.55,0.84,0.62,0.94,0.76,0.7041,0.282
3,ho,50,10,True,True,NaN,0.3,100,0.59,0.44,0.74,0.54,0.76,0.55,0.84,0.62,0.94,0.76,0.7041,0.281
4,ho,50,10,True,True,NaN,0.5,20,0.59,0.44,0.74,0.54,0.76,0.55,0.84,0.62,0.94,0.76,0.7041,0.288


=== ho 2단계 (fetch_k×rerank_k) 상위 5개 ===


,variant,alpha,rrf_k,fetch_k,rerank_k,use_reranker,use_cross_refs,max_cross_ref_tokens,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,ho,0.3,20,80,10,True,True,NaN,0.59,0.44,0.75,0.54,0.77,0.55,0.84,0.61,0.93,0.74,0.7065,0.608
1,ho,0.3,20,80,30,True,True,NaN,0.59,0.44,0.75,0.54,0.77,0.55,0.84,0.61,0.93,0.74,0.7065,0.621
2,ho,0.3,20,80,20,True,True,NaN,0.59,0.44,0.75,0.54,0.77,0.55,0.84,0.61,0.93,0.74,0.7065,0.608
3,ho,0.3,20,50,10,True,True,NaN,0.59,0.44,0.74,0.54,0.76,0.55,0.84,0.62,0.94,0.76,0.7041,0.283
4,ho,0.3,20,50,20,True,True,NaN,0.59,0.44,0.74,0.54,0.76,0.55,0.84,0.62,0.94,0.76,0.7041,0.271


=== ho 3단계 (use_cross_refs×max_cross_ref_tokens) 상위 5개 ===


,variant,alpha,rrf_k,fetch_k,rerank_k,use_reranker,use_cross_refs,max_cross_ref_tokens,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,ho,0.3,20,80,10,True,True,NaN,0.59,0.44,0.75,0.54,0.77,0.55,0.84,0.61,0.93,0.74,0.7065,0.647
1,ho,0.3,20,80,10,True,False,NaN,0.59,0.44,0.75,0.54,0.77,0.55,0.86,0.64,0.92,0.73,0.7059,0.001
2,ho,0.3,20,80,10,True,True,500.0,0.59,0.44,0.75,0.54,0.77,0.55,0.86,0.64,0.92,0.73,0.7059,0.845
3,ho,0.3,20,80,10,True,True,1000.0,0.59,0.44,0.75,0.54,0.77,0.55,0.86,0.64,0.92,0.73,0.7059,0.830
4,ho,0.3,20,80,10,True,True,2000.0,0.59,0.44,0.75,0.54,0.77,0.55,0.86,0.64,0.92,0.73,0.7059,0.850


In [33]:
from google.colab import files

for fname in [
    "eval_final_best.csv",
    "eval_stage1_alpha_rrf_jo.csv",
    "eval_stage2_fetchk_rerankk_jo.csv",
    "eval_stage1_alpha_rrf_ho.csv",
    "eval_stage2_fetchk_rerankk_ho.csv",
    "eval_stage3_crossref_ho.csv",
]:
    files.download(f"/content/{fname}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
!ps aux | grep yoonha_rag_eval_staged

root        2959  0.0  0.0   7376  3444 ?        S    13:26   0:00 /bin/bash -c ps aux | grep yoonha_rag_eval_staged
root        2961  0.0  0.0   6616  2476 ?        S    13:26   0:00 grep yoonha_rag_eval_staged


In [12]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.8Gi       5.1Gi       2.0Mi       4.8Gi       9.6Gi
Swap:             0B          0B          0B


In [13]:
!cat /proc/loadavg

1.58 1.07 0.51 1/345 3060


In [14]:
!find /content/qdrant_local -name "*.lock" -o -name "LOCK"

/content/qdrant_local/.lock


In [15]:
!python /content/yoonha_rag_eval_staged.py

python3: can't open file '/content/yoonha_rag_eval_staged.py': [Errno 2] No such file or directory


In [25]:
!rm /content/qdrant_local/.lock

In [26]:
!find /content/qdrant_local -iname "*lock*"